# Supplier Model Training

This notebook is the interactive counterpart to `train_model.py`. It creates an auditable supplier score from the procurement business rules, then compares Decision Tree, Random Forest, and XGBoost classifiers.

Run it from the project root, or leave the path setup cell enabled. Put the required four vendor CSVs in `../datasets/`.

In [1]:
from pathlib import Path
import sys

# Location of the ML project code
ROOT = Path(r"C:\Users\Soham\OneDrive\Documents\ChatGPT\EDI")

# Location of your datasets
DATA_DIR = Path(r"C:\EDI\datasets")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_supplier_dataset
from src.scoring import FEATURES, fit_score_bounds, calculate_supplier_score, score_to_class
from train_model import train

## Inspect cleaned supplier data
Each source is aggregated to one row per `VendorNumber` before merging, avoiding transaction-volume bias. Missing feature values are median-imputed.

In [2]:
%pip install xgboost pandas numpy matplotlib scikit-learn joblib seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
data = load_supplier_dataset(DATA_DIR)
data[['VendorNumber', *FEATURES]].head(), data.shape

(   VendorNumber  PurchasePrice   Quantity     Dollars    Freight  GrossProfit  \
 0             2      18.583077  25.230769  433.144615   6.770000 -2182.650000   
 1            54     105.070000   1.000000  105.070000   0.480000  -105.070000   
 2            60      16.230879   7.559105  122.636182  10.500571 -3064.676667   
 3           105      35.272807   5.824561  205.371930   1.559750  2000.305000   
 4           200       9.180000  16.500000  150.645000   1.031667   257.405000   
 
    ProfitMargin  StockTurnover  SalesToPurchaseRatio  
 0   -367.848944       1.162500              1.799205  
 1     22.391010       0.000000              0.000000  
 2     22.391010       0.681277              0.818672  
 3     24.007483       0.979688              1.316455  
 4      4.183257       0.701389              1.194047  ,
 (132, 18))

## Review the policy target
Bounds are normally fitted after the train/test split. This preview fits bounds only to display the score distribution; the training function below performs the leakage-safe split and fit automatically.

In [4]:
preview_bounds = fit_score_bounds(data)
data.assign(SupplierScore=calculate_supplier_score(data, preview_bounds), SupplierClass=lambda x: score_to_class(x['SupplierScore']))[['SupplierScore', 'SupplierClass']].describe(include='all')

,SupplierScore,SupplierClass
count,132.000000,132
unique,NaN,4
top,NaN,Average
freq,NaN,103
mean,64.525985,NaN
std,7.175123,NaN
min,37.210000,NaN
25%,61.842500,NaN
50%,66.435000,NaN
75%,67.950000,NaN


## Train, compare and save
The output includes `supplier_model.pkl`, a model-comparison table, classification report, confusion matrix, and feature-importance chart in `models/`. The highest weighted F1 score wins.

In [5]:
result = train(DATA_DIR, ROOT / 'models' / 'supplier_model.pkl')
result

{'best_model': 'xgboost',
 'model_path': 'C:\\Users\\Soham\\OneDrive\\Documents\\ChatGPT\\EDI\\models\\supplier_model.pkl',
 'comparison': [{'model': 'xgboost',
   'accuracy': 0.9259259259259259,
   'precision_weighted': 0.8904320987654322,
   'recall_weighted': 0.9259259259259259,
   'f1_weighted': 0.9078014184397163},
  {'model': 'random_forest',
   'accuracy': 0.8518518518518519,
   'precision_weighted': 0.9012345679012346,
   'recall_weighted': 0.8518518518518519,
   'f1_weighted': 0.8603215618719494},
  {'model': 'decision_tree',
   'accuracy': 0.7777777777777778,
   'precision_weighted': 0.8483245149911817,
   'recall_weighted': 0.7777777777777778,
   'f1_weighted': 0.8114478114478115}]}

In [6]:
import pandas as pd
pd.read_csv(ROOT / 'models' / 'model_comparison.csv')

,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,xgboost,0.925926,0.890432,0.925926,0.907801
1,random_forest,0.851852,0.901235,0.851852,0.860322
2,decision_tree,0.777778,0.848325,0.777778,0.811448


In [7]:
from predict import load_model, predict_supplier
import json

model = load_model(ROOT / "models" / "supplier_model.pkl")

with open(ROOT / "example_quote.json") as file:
    quotation = json.load(file)

prediction = predict_supplier(model, quotation)
prediction

{'supplier_name': 'ABC Steel Ltd',
 'supplier_score': 34.33,
 'rule_based_class': 'Poor',
 'supplier_class': 'Poor',
 'confidence_score': 82.21,
 'reason_for_recommendation': ['High profit margin',
  'High stock turnover',
  'Strong sales-to-purchase ratio'],
 'model': 'xgboost'}